In [ ]:
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as colormaps
import matplotlib.patches as patches

from mpl_toolkits.axes_grid1 import make_axes_locatable

import os

%matplotlib inline

In [ ]:
from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

In [ ]:
# import matplotlib.font_manager as font_manager
# font_manager._rebuild()

#print(matplotlib.get_cachedir())

# use LaTeX fonts in the plot
plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage[helvet]{sfmath}'

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

small_ticklabelsize=6
large_ticklabelsize=7
titlesize=20
fig_titlesize=26
axes_labelsize=8
axes_labelsize_small=8
subplot_labelsize=8
small_legendsize=6
colorbar_labelsize=8

mm = 1/25.4
l_w_ratio = 1/1.618

In [ ]:
output_dir = '/home/ben/Data/contact_maps/10000mono_testcase/'
output_label = 'replication_m0_ml0_t100000'

norm_flag = True

with open(output_dir+'pairs_'+output_label+'.bin','rb') as f:
    
    pairs = np.fromfile(f,dtype=np.int32,count=-1)
    
pairs = pairs.reshape((2,pairs.shape[0]//2),order='F').T
N_pairs = pairs.shape[0]

print('pairs')
print(pairs)
print(pairs.shape)
print('N_pairs')
print(N_pairs)

if norm_flag == True:

    with open(output_dir+'rval_norm_'+output_label+'.bin','rb') as f:

        vals = np.fromfile(f,dtype=np.float32,count=-1)
        
else:
    
    with open(output_dir+'rval_'+output_label+'.bin','rb') as f:

        vals = np.fromfile(f,dtype=np.float32,count=-1)
    
print('vals')
print(vals)
print('vals shape')
print(vals.shape)

N = pairs[-1,0]

pairs = pairs - 1

print(pairs)

print(N)

mat = np.zeros((N,N),dtype=np.float32)

# for i in range(N_pairs):
    
# #     j = pairs[i,0] - 1
# #     k = pairs[i,1] - 1
# #     mat[j,k] = vals[i]
    
#     mat[pairs[i,:]] = vals[i]

mat[pairs[:,0],pairs[:,1]] = vals

mat += (mat.T - np.diag(np.diagonal(mat)))

# mat = np.ma.log(mat)

In [ ]:
fig_size = [65,55]

fig = plt.figure(figsize=(fig_size[0]*mm,fig_size[1]*mm))

ax = plt.gca()

if norm_flag == True:
    map_norm = matplotlib.colors.Normalize(vmin=0.0, vmax=0.5)
else:
    map_norm = matplotlib.colors.Normalize(vmin=0.0, vmax=np.max(mat))

im = ax.imshow(mat,cmap='plasma',norm=map_norm)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)

cbar = fig.colorbar(im, cax=cax)
cbar.set_label(label=r'Contact Frequency', fontsize=6)
cbar.ax.tick_params(labelsize=4)

ax.set_xlabel(r'Locus', fontsize=8)
ax.set_ylabel(r'Locus', fontsize=8)

ax.tick_params(axis='x',labelsize=5)
ax.tick_params(axis='y',labelsize=5)

plt.tight_layout()

fig.savefig('./'+output_label+'_contact_map.pdf',dpi=300)
fig.savefig('./'+output_label+'_contact_map.png',dpi=300)

# plt.show()

In [ ]:
def contact_mat_alignment(N,mat):
    
    aligned_mat = np.zeros((N,N),dtype=mat.dtype)
    
    aligned_mat[0,:] = mat[0,:]
    
    for i in range(1,N):
        aligned_mat[i,:] = np.roll(mat[i,:],-i,axis=0)
    
    return aligned_mat

def multi_fits(ax,locus_size,regimes,contact_sum):
    
    for regime in reversed(regimes):

        X=np.ones((regime[1]-regime[0],2),dtype=np.double,order='F')

        z=(np.arange(regime[0],regime[1])*(locus_size/1000.0))
        if regime[0] == 0:
            z += 1

        X[:,1]=np.log(z)

        y=np.log(contact_sum[regime[0]:regime[1]])

        y=np.matmul(X.T,y)

        X=np.matmul(X.T,X)

        beta=np.matmul(np.linalg.inv(X),y)

        print(beta[1])

        y=beta[0]+beta[1]*np.log(z)

        y=np.exp(y)

        if regime[0] == 0:
            z[0]=0
            
        z=np.rint(z).astype(np.int64)
            
        temp_label=r'{:d}-{:d} kbp: '.format(z[0],z[-1])+'$s=-$'+"{:.3f}".format(-beta[1])
        
        ax.plot(z,y,ls="-",c='w',lw=2.0,alpha=0.65)
        ax.plot(z,y,ls="--",lw=1.5,alpha=0.9,label=temp_label)
    
    return

In [ ]:
aligned_mat = contact_mat_alignment(N,mat)

#print(aligned_mat)

contact_sum = np.mean(aligned_mat,axis=0)

#print(contact_sum)

In [ ]:
# fig_size = [55,55]
# fig_size = [87,55]
fig_size = [120,40]
# fig_size = [174,174]

fig = plt.figure(figsize=(fig_size[0]*mm,fig_size[1]*mm))

ax = plt.gca()

ax.set_xlabel(r'Genomic Distance [kbp] - $x$', fontsize=8)
ax.set_ylabel(r'Contact Probability - $P(x)$', fontsize=7)

temp_marker_style = dict(marker='.', markersize=8,fillstyle='none')

lower_lim = 1
upper_lim = 101
locus_size = 1000


za=(np.arange(lower_lim,upper_lim)*(locus_size/1000.0))

ax.plot(za,contact_sum[lower_lim:upper_lim],c='k',lw=1.0,alpha=0.35,**temp_marker_style)

regimes = [[1,11],[1,51],[1,101]]

multi_fits(ax,locus_size,regimes,contact_sum)

ax.set_xlim(left=lower_lim*(locus_size/1000.0),right=upper_lim*(locus_size/1000.0))

ax.set_yscale('log')
ax.set_xscale('log')

ax.legend(fontsize=7)

ax.tick_params(axis='x',labelsize=7)
ax.tick_params(axis='y',labelsize=7)

plt.tight_layout()

#plt.grid()

fig.savefig('./'+output_label+'_contact_law.pdf',dpi=300)
fig.savefig('./'+output_label+'_contact_law.png',dpi=300)